#### Importing Required Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

#### Plotting Defaults

In [3]:
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
})

COLORS = sns.color_palette("muted")
STATION_ORDER = ["S1_CNC_Machining", "S2_Welding", "S3_Assembly", "S4_Inspection", "S5_Packaging"]
STATION_LABELS = ["CNC Machining", "Welding", "Assembly", "Inspection", "Packaging"]

OUTPUT_DIR = "figures"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{name}.png", bbox_inches="tight")
    plt.show()
    plt.close()

#### Load Raw Data

In [8]:
prod_raw = pd.read_csv("data/production_log.csv")
down_raw = pd.read_csv("data/downtime_events.csv")
qual_raw = pd.read_csv("data/quality_events.csv")
print("=== RAW DATA SHAPES ===")
print(f"  Production log : {prod_raw.shape[0]:>8,} rows × {prod_raw.shape[1]} cols")
print(f"  Downtime events: {down_raw.shape[0]:>8,} rows × {down_raw.shape[1]} cols")
print(f"  Quality events : {qual_raw.shape[0]:>8,} rows × {qual_raw.shape[1]} cols")


=== RAW DATA SHAPES ===
  Production log :  211,750 rows × 12 cols
  Downtime events:      104 rows × 7 cols
  Quality events :    4,868 rows × 7 cols


#### Missing Values

In [13]:
print("\n=== MISSING VALUES (Production Log) ===")
missing = prod_raw.isnull().sum()
missing_pct = (prod_raw.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "pct": missing_pct})
print(missing_report[missing_report["missing_count"] > 0].to_string())


=== MISSING VALUES (Production Log) ===
                   missing_count    pct
shift                       4220   1.99
operator                  100351  47.39
queue_wait_min              4346   2.05
downtime_wait_min           4143   1.96
cycle_time_min              4266   2.01
start_min                   4163   1.97
end_min                     4158   1.96


#### Duplicate Values (rows identical on (part_id, station, status))

In [16]:
dup_key = ["part_id", "station", "status"]
n_dupes_prod = prod_raw.duplicated(subset=dup_key, keep="first").sum()
n_dupes_down = down_raw.duplicated(keep="first").sum()
n_dupes_qual = qual_raw.duplicated(subset=["part_id", "station", "defect_type"], keep="first").sum()
print("\n=== DUPLICATE RECORDS ===")
print(f"  Production log : {n_dupes_prod} duplicates "
      f"({n_dupes_prod / len(prod_raw) * 100:.2f}%)")
print(f"  Downtime events: {n_dupes_down} duplicates")
print(f"  Quality events : {n_dupes_qual} duplicates")


=== DUPLICATE RECORDS ===
  Production log : 1053 duplicates (0.50%)
  Downtime events: 0 duplicates
  Quality events : 24 duplicates


### **Cleaning decisions:**
##### - Drop duplicate rows (keep first occurrence).
##### - For missing numeric values (cycle times, queue waits), impute with station-level medians. Median is preferred over mean to avoid influence from outlier breakdown-related waits.
##### - Missing `shift` values are inferred from the `start_timestamp`.
##### - Missing `operator` values on automated stations are expected (no operator assigned); on manual stations they're flagged as unknown.

In [17]:
prod = prod_raw.drop_duplicates(subset=dup_key, keep="first").copy()

prod["start_dt"] = pd.to_datetime(prod["start_timestamp"], errors="coerce")
prod["end_dt"] = pd.to_datetime(prod["end_timestamp"], errors="coerce")
prod["date"] = prod["start_dt"].dt.date

def infer_shift(hour):
    if pd.isna(hour):
        return np.nan
    if 6 <= hour < 14:
        return 1
    elif 14 <= hour < 22:
        return 2
    else:
        return 3

missing_shift = prod["shift"].isna()
prod.loc[missing_shift, "shift"] = (prod.loc[missing_shift, "start_dt"].dt.hour.apply(infer_shift))
prod["shift"] = prod["shift"].astype("Int64")

numeric_cols = ["cycle_time_min", "queue_wait_min", "downtime_wait_min"]
for col in numeric_cols:
    station_medians = prod.groupby("station")[col].transform("median")
    prod[col] = prod[col].fillna(station_medians)

for col in numeric_cols:
    prod[col] = prod[col].fillna(prod[col].median())

manual_stations = ["S3_Assembly", "S4_Inspection", "S5_Packaging"]
prod.loc[prod["station"].isin(manual_stations) & prod["operator"].isna(), "operator"] = "UNKNOWN"

down = down_raw.drop_duplicates(keep="first").copy()
down["start_dt"] = pd.to_datetime(down["start_timestamp"], errors="coerce")
down["end_dt"] = pd.to_datetime(down["end_timestamp"], errors="coerce")

mask = down["duration_min"].isna() & down["start_dt"].notna() & down["end_dt"].notna()
down.loc[mask, "duration_min"] = ((down.loc[mask, "end_dt"] - down.loc[mask, "start_dt"]).dt.total_seconds() / 60)

qual = qual_raw.drop_duplicates(subset=["part_id", "station", "defect_type"], keep="first").copy()
qual["detection_dt"] = pd.to_datetime(qual["detection_time_timestamp"],errors="coerce")
 
print(f"\n=== CLEAN DATA SHAPES ===")
print(f"  Production log : {len(prod):>8,} rows  "
      f"(removed {len(prod_raw) - len(prod):,} dupes)")
print(f"  Downtime events: {len(down):>8,} rows  "
      f"(removed {len(down_raw) - len(down):,} dupes)")
print(f"  Quality events : {len(qual):>8,} rows  "
      f"(removed {len(qual_raw) - len(qual):,} dupes)")
print(f"  Remaining NaNs in production log: "
      f"{prod[numeric_cols].isna().sum().sum()}")


=== CLEAN DATA SHAPES ===
  Production log :  210,697 rows  (removed 1,053 dupes)
  Downtime events:      104 rows  (removed 0 dupes)
  Quality events :    4,844 rows  (removed 24 dupes)
  Remaining NaNs in production log: 0
